# Silver Layer — Prescriptions (`prescriptions`)

**Source:** `<source_schema>.prescriptions` (default `cdr_mimiciv.bronze.prescriptions`)
**Target:** `<target_schema>.prescriptions` (default `cdr_mimiciv.silver.prescriptions`)

**Objective:** Standardize prescription records while preserving patient and admission identifiers, medication orders, drug names, formulary codes, doses, routes, quantities, timing and other source clinical information.

**Grain:** One source prescription record (MIMIC-IV hosp `prescriptions`: one prescribed drug component of a pharmacy order; `pharmacy_id` links to `pharmacy`/`emar`, `poe_id`/`poe_seq` to `poe`). The official documentation does not define a unique key; `drug_type` (documented values MAIN/BASE/ADDITIVE, the component of the prescription the drug occupies) suggests one `pharmacy_id` may have several rows; the documentation does not state this explicitly, so it is an inference to be confirmed by profiling. Repeated medication orders or prescription identifiers are investigated rather than automatically removed, and uniqueness of any key is **profiled only, not asserted**.

**Silver rules:** Preserve every Bronze record and its clinical meaning. Standardize supported data types and non-clinical whitespace, validate identifiers and timestamps, inspect duplicates, and reconcile Bronze–Silver counts. No deletion, filtering, deduplication, imputation or aggregation.

**Publication:** The table is published with `CREATE OR REPLACE` (Delta) into `<target_schema>`; downstream Gold depends on it. Nothing is dropped. Hardcoded profiled counts are asserted only when widget `enforce_profiled_counts` is `true`.

**Cost note:** about 20M rows. Validation uses single-pass aggregates (no `exceptAll`, no repeated full scans).

**Dependencies:** none (reads Bronze only; no joins to other Silver tables).

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

SOURCE_TABLE = f"{BRONZE}.prescriptions"
TARGET_TABLE = f"{SILVER}.prescriptions"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)
print("Enforce profiled counts:", ENFORCE_PROFILED_COUNTS)

Source: cdr_mimiciv.bronze.prescriptions
Target: cdr_mimiciv.silver.prescriptions
Enforce profiled counts: True


In [0]:
# Load and inspect prescription records
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- pharmacy_id: integer (nullable = true)
 |-- poe_id: string (nullable = true)
 |-- poe_seq: integer (nullable = true)
 |-- order_provider_id: string (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- drug_type: string (nullable = true)
 |-- drug: string (nullable = true)
 |-- formulary_drug_cd: string (nullable = true)
 |-- gsn: string (nullable = true)
 |-- ndc: long (nullable = true)
 |-- prod_strength: string (nullable = true)
 |-- form_rx: string (nullable = true)
 |-- dose_val_rx: string (nullable = true)
 |-- dose_unit_rx: string (nullable = true)
 |-- form_val_disp: string (nullable = true)
 |-- form_unit_disp: string (nullable = true)
 |-- doses_per_24_hrs: integer (nullable = true)
 |-- route: string (nullable = true)



subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,starttime,stoptime,drug_type,drug,formulary_drug_cd,gsn,ndc,prod_strength,form_rx,dose_val_rx,dose_unit_rx,form_val_disp,form_unit_disp,doses_per_24_hrs,route
10000032,22595853,12775705,10000032-55,55,P85UQ1,2180-05-08T08:00:00.000Z,2180-05-07T22:00:00.000Z,MAIN,Furosemide,FURO40,008209,51079007320,40mg Tablet,null,40,mg,1,TAB,1,PO/NG
10000032,22595853,18415984,10000032-42,42,P23SJA,2180-05-07T02:00:00.000Z,2180-05-07T22:00:00.000Z,MAIN,Ipratropium Bromide Neb,IPRA2H,021700,487980125,2.5mL Vial,null,1,NEB,1,VIAL,4,IH
10000032,22595853,23637373,10000032-35,35,P23SJA,2180-05-07T01:00:00.000Z,2180-05-07T09:00:00.000Z,MAIN,Furosemide,FURO20,008208,51079007220,20mg Tablet,null,20,mg,1,TAB,1,PO/NG
10000032,22595853,26862314,10000032-41,41,P23SJA,2180-05-07T01:00:00.000Z,2180-05-07T01:00:00.000Z,MAIN,Potassium Chloride,MICROK10,001275,245004101,10mEq ER Tablet,null,40,mEq,4,TAB,1,PO
10000032,22595853,30740602,10000032-27,27,P23SJA,2180-05-07T00:00:00.000Z,2180-05-07T22:00:00.000Z,MAIN,Sodium Chloride 0.9% Flush,NACLFLUSH,null,0,10 mL Syringe,null,3,mL,0.3,SYR,3,IV
10000032,22595853,43092008,10000032-37,37,P23SJA,2180-05-07T01:00:00.000Z,2180-05-07T22:00:00.000Z,MAIN,Raltegravir,RALT400,063231,6022761,400 mg Tablet,null,400,mg,1,TAB,2,PO
10000032,22595853,46308055,10000032-56,56,P85UQ1,2180-05-08T08:00:00.000Z,2180-05-07T22:00:00.000Z,MAIN,Spironolactone,SPIR25,006817,63739054410,25mg Tablet,null,50,mg,2,TAB,1,PO/NG
10000032,22595853,48214508,10000032-39,39,P23SJA,2180-05-07T01:00:00.000Z,2180-05-07T22:00:00.000Z,MAIN,Acetaminophen,APAP500,004490,904198861,500mg Tablet,null,500,mg,1,TAB,null,PO/NG
10000032,22595853,71489491,10000032-23,23,P23SJA,2180-05-07T00:00:00.000Z,2180-05-07T22:00:00.000Z,MAIN,Influenza Vaccine Quadrivalent,INFL0.5LF,072514,19515089452,0.5 mL Syringe,null,0.5,mL,1,SYR,0,IM
10000032,22595853,73243129,10000032-33,33,P23SJA,2180-05-07T01:00:00.000Z,2180-05-07T22:00:00.000Z,MAIN,Albuterol Inhaler,ALBU17H,028090,173068224,8 g Inhaler,null,2,PUFF,0.02,INH,null,IH


In [0]:
%sql
Select distinct form_unit_disp
from cdr_mimiciv.bronze.prescriptions;

form_unit_disp
%
BULK
IMPLANT
PUMP
LIQ
null
AMP
NEB
Units VWF:RCo
SOLN


In [0]:
# Preserve all columns and standardize supported identifiers and timestamps
silver_df = bronze_df

long_columns = [
    "subject_id", "hadm_id", "pharmacy_id",
    "poe_seq"
]

timestamp_columns = [
    "starttime", "stoptime"
]

for c in long_columns:
    if c in silver_df.columns:
        silver_df = silver_df.withColumn(
            c, F.col(c).cast("long")
        )

for c in timestamp_columns:
    if c in silver_df.columns:
        silver_df = silver_df.withColumn(
            c, F.col(c).cast("timestamp")
        )

silver_df.printSchema()

# Ordered expected schema (Bronze columns, with the casts above applied)
EXPECTED_SCHEMA = {name: dtype for name, dtype in silver_df.dtypes}
for c in long_columns:
    if c in EXPECTED_SCHEMA:
        assert EXPECTED_SCHEMA[c] == "bigint", f"{c} not bigint"
for c in timestamp_columns:
    if c in EXPECTED_SCHEMA:
        assert EXPECTED_SCHEMA[c] == "timestamp", f"{c} not timestamp"

root
 |-- subject_id: long (nullable = true)
 |-- hadm_id: long (nullable = true)
 |-- pharmacy_id: long (nullable = true)
 |-- poe_id: string (nullable = true)
 |-- poe_seq: long (nullable = true)
 |-- order_provider_id: string (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- drug_type: string (nullable = true)
 |-- drug: string (nullable = true)
 |-- formulary_drug_cd: string (nullable = true)
 |-- gsn: string (nullable = true)
 |-- ndc: long (nullable = true)
 |-- prod_strength: string (nullable = true)
 |-- form_rx: string (nullable = true)
 |-- dose_val_rx: string (nullable = true)
 |-- dose_unit_rx: string (nullable = true)
 |-- form_val_disp: string (nullable = true)
 |-- form_unit_disp: string (nullable = true)
 |-- doses_per_24_hrs: integer (nullable = true)
 |-- route: string (nullable = true)



In [0]:
# Single pass over Bronze: row count, unsafe casts and reversed ranges
cast_checks = {
    "subject_id": "bigint",
    "hadm_id": "bigint",
    "pharmacy_id": "bigint",
    "poe_seq": "bigint",
    "starttime": "timestamp",
    "stoptime": "timestamp"
}
available_cast_checks = {
    c: dtype for c, dtype in cast_checks.items() if c in bronze_df.columns
}

bronze_aggs = [F.count("*").alias("rows")]
bronze_aggs += [
    F.sum(
        F.when(
            F.col(c).isNotNull()
            & F.expr(f"try_cast(`{c}` AS {dtype})").isNull(),
            1
        ).otherwise(0)
    ).alias(f"invalid_cast_{c}")
    for c, dtype in available_cast_checks.items()
]
if {"starttime", "stoptime"}.issubset(bronze_df.columns):
    bronze_aggs.append(
        F.sum(
            (
                F.expr("try_cast(starttime AS timestamp)").isNotNull()
                & F.expr("try_cast(stoptime AS timestamp)").isNotNull()
                & (
                    F.expr("try_cast(stoptime AS timestamp)")
                    < F.expr("try_cast(starttime AS timestamp)")
                )
            ).cast("long")
        ).alias("reversed_times")
    )
if "drug_type" in bronze_df.columns:
    bronze_aggs.append(
        F.sum(F.col("drug_type").isNull().cast("long")).alias("null_drug_type")
    )

bronze_stats = bronze_df.agg(*bronze_aggs).first().asDict()
display(spark.createDataFrame([(k, int(v or 0)) for k, v in bronze_stats.items()], ["metric", "value"]))

bronze_count = bronze_stats["rows"]
bronze_invalid_times = int(bronze_stats.get("reversed_times") or 0)
cast_issues = {k: v for k, v in bronze_stats.items() if k.startswith("invalid_cast_") and v}

print("Bronze rows:", bronze_count)
print("Bronze reversed timestamps:", bronze_invalid_times)

metric,value
rows,20292611
invalid_cast_subject_id,0
invalid_cast_hadm_id,0
invalid_cast_pharmacy_id,0
invalid_cast_poe_seq,0
invalid_cast_starttime,0
invalid_cast_stoptime,0
reversed_times,816994
null_drug_type,0


Bronze rows: 20292611
Bronze reversed timestamps: 816994


In [0]:
# Single pass over Silver: row count, missingness and reversed ranges
critical_columns = [
    "subject_id", "hadm_id", "pharmacy_id",
    "poe_id", "starttime", "stoptime", "drug"
]
available_critical_columns = [
    c for c in critical_columns if c in silver_df.columns
]

has_times = {"starttime", "stoptime"}.issubset(silver_df.columns)
reversed_cond = (
    F.col("starttime").isNotNull()
    & F.col("stoptime").isNotNull()
    & (F.col("stoptime") < F.col("starttime"))
) if has_times else F.lit(False)
difference_hours = (
    (F.unix_timestamp("starttime") - F.unix_timestamp("stoptime")) / 3600.0
) if has_times else F.lit(None).cast("double")

silver_aggs = [F.count("*").alias("rows")]
silver_aggs += [
    F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}")
    for c in available_critical_columns
]
silver_aggs += [
    F.sum(reversed_cond.cast("long")).alias("reversed_times"),
    F.min(F.when(reversed_cond, difference_hours)).alias("reversed_min_hours"),
    F.percentile_approx(F.when(reversed_cond, difference_hours), 0.5).alias("reversed_median_hours"),
    F.max(F.when(reversed_cond, difference_hours)).alias("reversed_max_hours"),
]

silver_stats = silver_df.agg(*silver_aggs).first().asDict()
display(spark.createDataFrame([(k, None if v is None else float(v)) for k, v in silver_stats.items()], ["metric", "value"]))

silver_count = silver_stats["rows"]
silver_invalid_times = int(silver_stats["reversed_times"] or 0)

print("Silver rows:", silver_count)
print("Silver reversed timestamps (preserved, review only):", silver_invalid_times)

metric,value
rows,2.0292611E7
null_subject_id,0.0
null_hadm_id,0.0
null_pharmacy_id,0.0
null_poe_id,184441.0
null_starttime,21890.0
null_stoptime,31436.0
null_drug,1.0
reversed_times,816994.0
reversed_min_hours,1.0


Silver rows: 20292611
Silver reversed timestamps (preserved, review only): 816994


In [0]:
# Inspect exact duplicate prescriptions without removing records (single pass)
dup_row = (
    silver_df
    .groupBy(*silver_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.coalesce(F.sum("count"), F.lit(0)).alias("records_in_duplicate_groups"),
        F.coalesce(F.max("count"), F.lit(0)).alias("maximum_group_size")
    )
    .first()
)
duplicate_count = dup_row["duplicate_groups"]

print("Exact duplicate prescription groups:", duplicate_count)
print("Records in duplicate groups:", dup_row["records_in_duplicate_groups"])
print("Maximum group size:", dup_row["maximum_group_size"])

Exact duplicate prescription groups: 0
Records in duplicate groups: 0
Maximum group size: 0


In [0]:
# Review rows with missing drug_type (documented NOT NULL in MIMIC-IV); sample only
if "drug_type" in bronze_df.columns:
    display(bronze_df.filter(F.col("drug_type").isNull()).limit(20))

subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,starttime,stoptime,drug_type,drug,formulary_drug_cd,gsn,ndc,prod_strength,form_rx,dose_val_rx,dose_unit_rx,form_val_disp,form_unit_disp,doses_per_24_hrs,route


In [0]:
# Pre-publish validation (row preservation, safe casts, required identifiers)
required_columns = ["subject_id", "hadm_id"]
issues = {
    f"null_{c}": silver_stats[f"null_{c}"]
    for c in required_columns
    if f"null_{c}" in silver_stats and silver_stats[f"null_{c}"] != 0
}

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"
assert not issues, f"Critical prescription quality issues: {issues}"
assert not cast_issues, f"Unsafe type conversions: {cast_issues}"
assert bronze_invalid_times == silver_invalid_times, (
    "Timestamp anomaly counts differ between Bronze and Silver"
)

if ENFORCE_PROFILED_COUNTS:
    assert duplicate_count == 0, "Exact duplicate prescriptions require review"

print("All pre-publish checks passed.")
print("Invalid timestamp ranges (review only):", silver_invalid_times)

All pre-publish checks passed.
Invalid timestamp ranges (review only): 816994


In [0]:
%sql
drop table cdr_mimiciv.silver.prescriptions;

In [0]:
# Publish all validated records (CREATE OR REPLACE), including documented source timestamp anomalies
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)
print("Preserved source timestamp anomalies:", silver_invalid_times)

Published: cdr_mimiciv.silver.prescriptions
Preserved source timestamp anomalies: 816994


In [0]:
# Verify the published Silver prescriptions table (single aggregate pass)
published_df = spark.table(TARGET_TABLE)

published_schema = {name: dtype for name, dtype in published_df.dtypes}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from expected: {published_schema}"
)

published_stats = published_df.agg(
    F.count("*").alias("rows"),
    *[
        F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}")
        for c in required_columns if c in published_df.columns
    ]
).first().asDict()

print("Bronze rows:", bronze_count)
print("Published rows:", published_stats["rows"])

assert published_stats["rows"] == bronze_count, "Published row-count mismatch"
assert all(v == 0 for k, v in published_stats.items() if k.startswith("null_")), (
    "Published required identifiers contain NULLs"
)

print("Silver prescriptions table verified successfully.")

Bronze rows: 20292611
Published rows: 20292611
Silver prescriptions table verified successfully.


In [0]:
dbutils.notebook.exit("prescriptions:OK")